# oWAR Overview Dashboard - Current Season Projections

**Purpose:** All-in-one dashboard for 2025 pitcher and hitter WAR projections

**Last Updated:** 2025-10-06

---

## Features
- Interactive scatter plots (WAR vs IP/PA)
- Featured player tables with rankings
- Two-way player support (Shohei Ohtani)
- Rest of season (ROS) projections

In [ ]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path

# Add project root to path
project_root = Path('.').absolute().parent.parent
sys.path.insert(0, str(project_root))

from new_pipeline.notebooks.shared.pipeline_runner import (
    load_current_season_data,
    run_data_pipeline,
    generate_predictions
)
from new_pipeline.notebooks.shared.plotting_utils import create_war_scatter
from new_pipeline.notebooks.shared.table_utils import (
    create_featured_table,
    handle_two_way_player
)
from new_pipeline.models import PitcherRoleEnsemble, HitterEnsemble

print("Imports successful!")

In [ ]:
# Cell 2: Load Current Season Data

print("Loading 2025 current season data...")

# Load raw data
pitcher_raw = load_current_season_data('pitcher', year=2025)
hitter_raw = load_current_season_data('hitter', year=2025)

print(f"Loaded {len(pitcher_raw)} pitchers (raw)")
print(f"Loaded {len(hitter_raw)} hitters (raw)")

# Run pipelines
print("\nRunning data pipelines...")
pitcher_processed = run_data_pipeline(pitcher_raw, player_type='pitcher')
hitter_processed = run_data_pipeline(hitter_raw, player_type='hitter')

print(f"Processed {len(pitcher_processed)} pitchers (qualified)")
print(f"Processed {len(hitter_processed)} hitters (qualified)")

# Load pre-trained models (from integration test or previous training)
# NOTE: Replace with actual model loading when models are saved
print("\nTraining models on current data...")

# Train pitcher model
from new_pipeline.common.constants import PITCHER_MODEL_FEATURES
pitcher_model = PitcherRoleEnsemble()

# Create role labels
pitcher_processed['GS_per_G'] = pitcher_processed['GS'] / pitcher_processed['G'].replace(0, 1)
def get_role(row):
    if row['GS_per_G'] > 0.7:
        return 'starter'
    elif row['GS_per_G'] < 0.1:
        return 'reliever'
    else:
        return 'swing'

roles = pitcher_processed.apply(get_role, axis=1).values
X_pitcher = pitcher_processed[PITCHER_MODEL_FEATURES].values
y_pitcher = pitcher_processed['WAR_per_162'].values
pitcher_model.fit(X_pitcher, y_pitcher, roles)

# Train hitter model
from new_pipeline.common.constants import HITTER_MODEL_FEATURES
hitter_model = HitterEnsemble()
X_hitter = hitter_processed[HITTER_MODEL_FEATURES].values
y_hitter = hitter_processed['WAR_per_600'].values
hitter_model.fit(X_hitter, y_hitter)

print("Models trained!")

# Generate predictions
print("\nGenerating predictions...")
pitcher_predictions = generate_predictions(
    pitcher_processed,
    pitcher_model,
    player_type='pitcher'
)
hitter_predictions = generate_predictions(
    hitter_processed,
    hitter_model,
    player_type='hitter'
)

print(f"\nGenerated predictions for {len(pitcher_predictions)} pitchers")
print(f"Generated predictions for {len(hitter_predictions)} hitters")
print("\nReady for visualization!")

In [ ]:
# Cell 3: Pitcher Projections Scatter

# Add pitcher type for coloring
pitcher_predictions['Type'] = pitcher_predictions.apply(get_role, axis=1)
pitcher_predictions['Type'] = pitcher_predictions['Type'].map({
    'starter': 'Starter',
    'reliever': 'Reliever',
    'swing': 'Swing'
})

fig_pitchers = create_war_scatter(
    df=pitcher_predictions,
    player_type='pitcher',
    title="2025 Pitcher WAR Projections",
    color_by='Type',
    hover_data=['Name', 'Team', 'Type', 'IP', 'Current_WAR', 'ROS_WAR', 'Total_Projected_WAR']
)

fig_pitchers.show()

In [ ]:
# Cell 4: Hitter Projections Scatter

# Add position placeholder (simplified for overview)
# In real implementation, get from Primary_Position feature
hitter_predictions['Pos'] = 'IF'  # Placeholder

fig_hitters = create_war_scatter(
    df=hitter_predictions,
    player_type='hitter',
    title="2025 Hitter WAR Projections",
    color_by='Pos',
    hover_data=['Name', 'Team', 'Pos', 'PA', 'Current_WAR', 'ROS_WAR', 'Total_Projected_WAR']
)

fig_hitters.show()

In [ ]:
# Cell 5: Featured Pitchers Table

# Define featured pitchers (manually curated list)
featured_pitchers = [
    'Tarik Skubal',
    'Zack Wheeler',
    'Corbin Burnes',
    'Chris Sale',
    'Emmanuel Clase'
]

# Check if Shohei Ohtani is in both datasets (two-way player)
two_way_data = None
if 'Shohei Ohtani' in pitcher_predictions['Name'].values and 'Shohei Ohtani' in hitter_predictions['Name'].values:
    featured_pitchers.append('Shohei Ohtani')
    
    ohtani_pitcher = pitcher_predictions[pitcher_predictions['Name'] == 'Shohei Ohtani'].iloc[0]
    ohtani_hitter = hitter_predictions[hitter_predictions['Name'] == 'Shohei Ohtani'].iloc[0]
    
    two_way_data = {
        'Shohei Ohtani': handle_two_way_player(
            pitcher_war={'current': ohtani_pitcher['Current_WAR'],
                        'ROS': ohtani_pitcher['ROS_WAR'],
                        'total': ohtani_pitcher['Total_Projected_WAR']},
            hitter_war={'current': ohtani_hitter['Current_WAR'],
                       'ROS': ohtani_hitter['ROS_WAR'],
                       'total': ohtani_hitter['Total_Projected_WAR']}
        )
    }

# Create table
pitcher_table = create_featured_table(
    df=pitcher_predictions,
    player_names=featured_pitchers,
    player_type='pitcher',
    two_way_data=two_way_data
)

print("=" * 70)
print("FEATURED PITCHERS")
print("=" * 70)
print(pitcher_table)

In [ ]:
# Cell 6: Featured Hitters Table

# Define featured hitters (manually curated list)
featured_hitters = [
    'Aaron Judge',
    'Juan Soto',
    'Bobby Witt Jr.',
    'Freddie Freeman',
    'Mookie Betts'
]

if 'Shohei Ohtani' in hitter_predictions['Name'].values:
    featured_hitters.append('Shohei Ohtani')

# Create table (two_way_data already defined from Cell 5)
hitter_table = create_featured_table(
    df=hitter_predictions,
    player_names=featured_hitters,
    player_type='hitter',
    two_way_data=two_way_data
)

print("=" * 70)
print("FEATURED HITTERS")
print("=" * 70)
print(hitter_table)